# ML dataset source and API provenance audit

**Question:** Can the source data and external requests behind the maritime freight ML database be verified against the original publishers?

**Scope:** Read-only inspection of `D:\Internproj` source files and CSVs. The notebook checks local grain, price-route duplication, LSCI acquisition code, and NOAA precipitation-grid compatibility. It makes no external API requests and does not modify data.

**Audit date:** 2026-10-06. Paths below assume the project is at `D:\Internproj`.


In [1]:
from pathlib import Path
import ast
import pandas as pd
import numpy as np

PROJECT = Path(r"D:\Internproj")
PRICE_FILE = PROJECT / "Index" / "xeneta" / "xeneta_xsi_combined.csv"
LSCI_FILE = PROJECT / "oprational costs" / "unctad_lsci" / "unctad_lsci_2015_2024.csv"
LSCI_SCRIPT = PROJECT / "oprational costs" / "unctad_lsci" / "fetch_lsci.py"
ENGINEERED_FILE = PROJECT / "new dataset" / "engineered_ml_dataset.csv"
CLEAN_FILE = PROJECT / "New ML" / "clean_engineered_dataset.csv"
PRATE_RAW = PROJECT / "weatherdata" / "datasets" / "noaa_raw" / "prate" / "prate.2025.csv"


## Dataset grain and duplicated routes

The intended grain is one row per `(Route, Date)`. This cell reports duplicate keys, time span, route coverage, and exact price equality between reverse-direction route pairs.


In [2]:
prices = pd.read_csv(PRICE_FILE, parse_dates=["Date"])
summary = {
    "rows": len(prices),
    "columns": list(prices.columns),
    "duplicate_route_date_keys": int(prices.duplicated(["Route", "Date"]).sum()),
    "date_min": prices["Date"].min(),
    "date_max": prices["Date"].max(),
    "routes": prices.groupby("Route").size().to_dict(),
}
wide = prices.pivot(index="Date", columns="Route", values="Price_USD")
mirror_pairs = [
    ("XSICFENE_FarEast_NorthEurope", "XSICNEFE_NorthEurope_FarEast"),
    ("XSICFEUW_FarEast_USWestCoast", "XSICUWFE_USWestCoast_FarEast"),
]
mirror_results = []
for route_a, route_b in mirror_pairs:
    pair = wide[[route_a, route_b]].dropna()
    mirror_results.append({
        "route_a": route_a,
        "route_b": route_b,
        "overlap_dates": len(pair),
        "equal_prices": int(pair[route_a].eq(pair[route_b]).sum()),
        "correlation": pair[route_a].corr(pair[route_b]),
    })
summary, pd.DataFrame(mirror_results)


({'rows': 9800,
  'columns': ['Date', 'Price_USD', 'Route'],
  'duplicate_route_date_keys': 0,
  'date_min': Timestamp('2015-01-05 00:00:00'),
  'date_max': Timestamp('2026-04-15 00:00:00'),
  'routes': {'XSICFENE_FarEast_NorthEurope': 2829,
   'XSICFEUW_FarEast_USWestCoast': 2071,
   'XSICNEFE_NorthEurope_FarEast': 2829,
   'XSICUWFE_USWestCoast_FarEast': 2071}},
                         route_a                       route_b  overlap_dates  \
 0  XSICFENE_FarEast_NorthEurope  XSICNEFE_NorthEurope_FarEast           2829   
 1  XSICFEUW_FarEast_USWestCoast  XSICUWFE_USWestCoast_FarEast           2071   
 
    equal_prices  correlation  
 0          2829          1.0  
 1          2071          1.0  )

## LSCI: check whether the claimed fetcher makes a request

The project labels `fetch_lsci.py` a fetcher. The following AST scan finds actual calls through `requests.*`, `urllib.*`, or `urlopen`, and lists literal annual/quarterly table assignments. It does not execute the script.


In [3]:
lsci_source = LSCI_SCRIPT.read_text(encoding="utf-8")
lsci_tree = ast.parse(lsci_source)
network_calls = []
for node in ast.walk(lsci_tree):
    if isinstance(node, ast.Call):
        f = node.func
        if isinstance(f, ast.Attribute) and isinstance(f.value, ast.Name):
            if f.value.id in {"requests", "urllib", "urlopen"}:
                network_calls.append(ast.unparse(f))
        elif isinstance(f, ast.Name) and f.id in {"urlopen"}:
            network_calls.append(ast.unparse(f))

literal_tables = [
    node.targets[0].id
    for node in ast.walk(lsci_tree)
    if isinstance(node, ast.Assign)
    and node.targets
    and isinstance(node.targets[0], ast.Name)
    and node.targets[0].id in {"annual_data_2015_2019", "quarterly_data_2020_2024"}
]
lsci = pd.read_csv(LSCI_FILE)
lsci_summary = {
    "actual_network_calls_in_fetcher": network_calls,
    "hard_coded_tables": literal_tables,
    "csv_rows": len(lsci),
    "countries": lsci["Country"].nunique(),
    "period_counts": lsci["Quarter"].value_counts().to_dict(),
    "2024_q4_china": lsci.loc[(lsci.Year == 2024) & (lsci.Quarter == "Q4") & (lsci.Country == "China"), "LSCI_Score"].tolist(),
}
lsci_summary


{'actual_network_calls_in_fetcher': [],
 'hard_coded_tables': ['annual_data_2015_2019', 'quarterly_data_2020_2024'],
 'csv_rows': 500,
 'countries': 20,
 'period_counts': {'Annual': 100, 'Q1': 100, 'Q2': 100, 'Q3': 100, 'Q4': 100},
 '2024_q4_china': [128.3]}

## NOAA precipitation coordinate compatibility

`process_weather.py` / `build_dataset.py` map exact `(lat, lon)` tuples from a 2.5° grid. This checks sample PRATE coordinates against that map and verifies whether precipitation columns reached the engineered dataset. It uses only a bounded CSV sample to avoid loading the hundreds-of-megabytes raw file.


In [4]:
latitudes = np.arange(-90, 90.1, 2.5)
longitudes = np.arange(0, 360, 2.5)
zone_coords = set()
for lat in latitudes:
    for lon in longitudes:
        if (5 <= lat <= 25 and 50 <= lon <= 75) or (-30 <= lat <= 5 and 50 <= lon <= 100):
            zone_coords.add((lat, lon))
        elif (30 <= lat <= 45 and (lon >= 355 or lon <= 35)) or (50 <= lat <= 65 and (lon >= 355 or lon <= 15)):
            zone_coords.add((lat, lon))
        elif 12 <= lat <= 30 and 32.5 <= lon <= 45:
            zone_coords.add((lat, lon))
        elif 0 <= lat <= 25 and 100 <= lon <= 120:
            zone_coords.add((lat, lon))
        elif 20 <= lat <= 55 and (120 <= lon <= 180 or 180 <= lon <= 240):
            zone_coords.add((lat, lon))
        elif 20 <= lat <= 60 and 290 <= lon <= 350:
            zone_coords.add((lat, lon))

prate_sample = pd.read_csv(PRATE_RAW, usecols=["lat", "lon"], nrows=100_000)
prate_sample["lat"] = pd.to_numeric(prate_sample["lat"], errors="coerce")
prate_sample["lon"] = pd.to_numeric(prate_sample["lon"], errors="coerce")
prate_sample_coords = set(map(tuple, prate_sample[["lat", "lon"]].drop_duplicates().to_numpy()))
prate_engineered = pd.read_csv(ENGINEERED_FILE, nrows=1).columns
precip_columns = [c for c in prate_engineered if "prate" in c.lower() or "precip" in c.lower()]
{
    "sample_coordinate_pairs": len(prate_sample_coords),
    "sample_pairs_matching_exact_2_5_degree_zone_grid": len(prate_sample_coords & zone_coords),
    "sample_latitude_examples": sorted(prate_sample["lat"].dropna().unique())[:8],
    "sample_longitude_examples": sorted(prate_sample["lon"].dropna().unique())[:8],
    "precipitation_columns_in_engineered_dataset": precip_columns,
}


{'sample_coordinate_pairs': 18048,
 'sample_pairs_matching_exact_2_5_degree_zone_grid': 0,
 'sample_latitude_examples': [np.float64(-88.5419998),
  np.float64(-86.6530991),
  np.float64(-84.7531967),
  np.float64(-82.8507996),
  np.float64(-80.9473038),
  np.float64(-79.0435028),
  np.float64(-77.1393967),
  np.float64(-75.2350998)],
 'sample_longitude_examples': [np.float64(0.0),
  np.float64(1.875),
  np.float64(3.75),
  np.float64(5.625),
  np.float64(7.5),
  np.float64(9.375),
  np.float64(11.25),
  np.float64(13.125)],
 'precipitation_columns_in_engineered_dataset': []}

## Original-source facts to verify outside this local-file notebook

- [UNCTAD LSCI metadata](https://unctadstat.unctad.org/datacentre/reportInfo/US.LSCI): the 2024 revision sets 100 to the average country connectivity in Q1 2023, applies the rescaling to the history from Q1 2006, and says monthly LSCI started in 2025.
- [UNCTAD methodology revision](https://unctad.org/news/new-context-calls-changing-how-we-measure-maritime-connectivity): defines the new reference scale and explains why old and revised values are not directly comparable.
- [Xeneta XSI-C](https://xsi.xeneta.com/): distinct directional lanes; daily median prices for 40-foot containers; historical/daily access is via Xeneta/Compass. Values in this project's CSV have no export metadata.
- [NOAA NCEP/NCAR Reanalysis 1](https://psl.noaa.gov/data/gridded/data.ncep.reanalysis.html): original reanalysis catalog and coverage. The project's raw-grid derivation should be reproducible from its source files and coordinate system.
- [NOAA IBTrACS v04r01](https://www.ncei.noaa.gov/products/international-best-track-archive): official archive, version, data access, and column documentation.

These links establish source definitions and access—not provenance of a local CSV. A matching value without a dated raw download, query parameters, version, and checksum is not enough to prove that the stored row came from that source.


## Current audit conclusion

Local checks show exact reverse-direction price duplication and a UNCTAD-labeled script that should be checked for actual network calls versus literal input tables. The original-source documentation identifies the appropriate scale and instrument definitions, but local CSV provenance is not established by those documents. Keep the raw provider extract, request URL/parameters, retrieved-at timestamp, provider release/version, checksum, and transformation manifest together before using the features for model claims.
